# Hoffbauer Antiprimes - Berechnung mit SageMath

Dieses Notebook implementiert die Hoffbauer-Dirac-Operator-Berechnungen für Antiprimes mit SageMath 10.8.

In [ ]:
# SageMath Imports
from sage.all import *
import pandas
# Sicherstellen, dass CC verfügbar ist (ComplexField für komplexe Zahlen)
if 'CC' not in dir():
    CC = ComplexField(53)  # Standard-Genauigkeit für komplexe Zahlen

# Standard Python-Bibliotheken
import numpy as np
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("Hinweis: pandas nicht verfügbar, verwende einfache Listen/Dictionaries")

try:
    import matplotlib.pyplot as plt
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("Hinweis: matplotlib nicht verfügbar, Visualisierungen werden übersprungen")

## Familien-Definitionen

## Definition: Quadrupel von Primzahlen

**Definition eines Quadrupels:**
- Ein Quadrupel ist das **Produkt von vier Primzahlen**
- Jede Primzahl muss aus einer **anderen Familie** kommen
- Das Produkt besteht aus **genau einer Primzahl** aus jeder der Familien **E, A, B, C**

**Formal:**
- p_e ∈ Familie E (Primzahlen ≡ 1 mod 12)
- p_a ∈ Familie A (Primzahlen ≡ 5 mod 12)  
- p_b ∈ Familie B (Primzahlen ≡ 7 mod 12)
- p_c ∈ Familie C (Primzahlen ≡ 11 mod 12)
- **Produkt: P = p_e * p_a * p_b * p_c**

**Evaluationssequenz (wie implementiert):**
1. **Sammle Primzahlen nach Familien**: Alle Primzahlen werden nach ihren Restklassen mod 12 gruppiert
2. **Generiere alle Kombinationen**: Für jede Kombination (p_e, p_a, p_b, p_c) mit:
   - p_e aus Familie E
   - p_a aus Familie A
   - p_b aus Familie B
   - p_c aus Familie C
3. **Berechne das Produkt**: P = p_e * p_a * p_b * p_c
4. **Optional - Lagrange-Zerlegung**: Finde e, a, b, c so dass P = e² + a² + b² + c² (zur Verifikation/Analyse)

In [ ]:
# ---------- Familien (deine Definition) ----------
def fam_mod12(p):
    r = int(p % 12)
    if r == 1:  return "e"
    if r == 5:  return "a"
    if r == 7:  return "b"
    if r == 11: return "c"
    return "x"

def fam_phase(p):
    mp = {"e":0.0, "a":np.pi/2, "b":np.pi, "c":3*np.pi/2, "x":0.0}
    return mp[fam_mod12(p)]

In [ ]:
# Funktionen für Quadrupel von Primzahlen aus Familien E, A, B, C
# Hinweis: Stellen Sie sicher, dass Zelle 1 (Imports) zuerst ausgeführt wurde!
from sage.all import Integer, is_squarefree, is_prime, next_prime, sqrt
from itertools import product

def get_family_from_prime(p):
    """Gibt die Familie einer Primzahl zurück (e, a, b, c oder None)"""
    r = int(p % 12)
    if r == 1:  return "e"
    if r == 5:  return "a"
    if r == 7:  return "b"
    if r == 11: return "c"
    return None

def get_primes_by_family(max_prime):
    """Sammelt Primzahlen nach Familien bis max_prime"""
    families = {"e": [], "a": [], "b": [], "c": []}
    p = 2
    while p <= max_prime:
        fam = get_family_from_prime(p)
        if fam:
            families[fam].append(p)
        p = next_prime(p)
    return families

def four_squares_decomposition(n):
    """
    Zerlegt eine Zahl n in eine Summe von 4 Quadraten: n = e² + a² + b² + c²
    Verwendet SageMath's is_sum_of_four_squares falls verfügbar.
    """
    try:
        n_int = Integer(n)
        # Versuche SageMath-Methode
        if hasattr(n_int, 'is_sum_of_four_squares'):
            result = n_int.is_sum_of_four_squares(ext=True)
            if result:
                return list(result)
        
        # Fallback: Brute-Force für kleinere Zahlen
        n_val = int(n)
        sqrt_n = int(sqrt(n_val))
        
        for e in range(sqrt_n + 1):
            e_sq = e * e
            if e_sq > n_val:
                break
            for a in range(sqrt_n + 1):
                a_sq = a * a
                if e_sq + a_sq > n_val:
                    break
                for b in range(sqrt_n + 1):
                    b_sq = b * b
                    if e_sq + a_sq + b_sq > n_val:
                        break
                    c_sq = n_val - e_sq - a_sq - b_sq
                    if c_sq >= 0:
                        c = int(sqrt(c_sq))
                        if c * c == c_sq:
                            return [e, a, b, c]
        return None
    except:
        return None

def generate_quadruples(max_P=1000000):
    """
    Generiert alle Quadrupel von Primzahlen (p_e, p_a, p_b, p_c) wobei:
    - p_e ∈ Familie E (Primzahlen ≡ 1 mod 12)
    - p_a ∈ Familie A (Primzahlen ≡ 5 mod 12)
    - p_b ∈ Familie B (Primzahlen ≡ 7 mod 12)
    - p_c ∈ Familie C (Primzahlen ≡ 11 mod 12)
    - Produkt: P = p_e * p_a * p_b * p_c ≤ max_P
    - Lagrange-Zerlegung: P = e² + a² + b² + c²
    
    Args:
        max_P: Maximales Produkt P (default: 1.000.000)
    """
    # Bestimme maximale Primzahl (ungefähr 4. Wurzel von max_P)
    max_prime = int(max_P ** 0.25) + 100
    
    print(f"Berechne Quadrupel von Primzahlen bis P = {max_P:,}")
    print(f"Maximale Primzahl pro Familie: ~{max_prime}")
    print("=" * 60)
    
    # Sammle Primzahlen nach Familien
    print("Sammle Primzahlen nach Familien...")
    families = get_primes_by_family(max_prime)
    
    print(f"  Familie E: {len(families['e'])} Primzahlen (z.B. {families['e'][:10]}...)")
    print(f"  Familie A: {len(families['a'])} Primzahlen (z.B. {families['a'][:10]}...)")
    print(f"  Familie B: {len(families['b'])} Primzahlen (z.B. {families['b'][:10]}...)")
    print(f"  Familie C: {len(families['c'])} Primzahlen (z.B. {families['c'][:10]}...)")
    
    results = []
    total_checked = 0
    
    print(f"\nGeneriere Quadrupel von Primzahlen...")
    
    # Iteriere durch alle Kombinationen von Primzahlen
    for p_e in families['e']:
        for p_a in families['a']:
            P_temp = p_e * p_a
            if P_temp > max_P:
                break
            
            for p_b in families['b']:
                P_temp2 = P_temp * p_b
                if P_temp2 > max_P:
                    break
                
                for p_c in families['c']:
                    P = P_temp2 * p_c
                    total_checked += 1
                    
                    if P > max_P:
                        break
                    
                    # P ist automatisch quadratfrei (Produkt verschiedener Primzahlen)
                    # Finde Lagrange-Zerlegung: P = e² + a² + b² + c²
                    lagrange = four_squares_decomposition(P)
                    
                    if lagrange:
                        e, a, b, c = lagrange
                        results.append({
                            "P": P,  # Produkt der 4 Primzahlen
                            "p_e": p_e,  # Primzahl aus Familie E
                            "p_a": p_a,  # Primzahl aus Familie A
                            "p_b": p_b,  # Primzahl aus Familie B
                            "p_c": p_c,  # Primzahl aus Familie C
                            "e": e,  # Lagrange-Komponente
                            "a": a,  # Lagrange-Komponente
                            "b": b,  # Lagrange-Komponente
                            "c": c,  # Lagrange-Komponente
                            "L": P,  # Lagrange-Zahl (gleich P)
                            "check": e*e + a*a + b*b + c*c  # Verifikation
                        })
                    
                    # Fortschrittsanzeige
                    if total_checked % 10000 == 0:
                        print(f"  {total_checked:,} Kombinationen geprüft, {len(results):,} gültige Quadrupel gefunden...")
    
    print(f"\n{'=' * 60}")
    print(f"Fertig: {total_checked:,} Kombinationen geprüft")
    print(f"{len(results):,} Quadrupel von Primzahlen gefunden")
    print(f"{'=' * 60}")
    
    return results

: 

### Berechnung aller Quadrupel bis 1.000.000

In [ ]:
# Stelle sicher, dass pandas importiert ist
try:
    pd
except NameError:
    print("Hinweis: pandas wird jetzt importiert (Zelle 1 sollte zuerst ausgeführt werden)")
    import pandas as pd

# Berechne alle Quadrupel von Primzahlen bis P = 1.000.000
results = generate_quadruples(max_P=1000000)

# Erstelle DataFrame für bessere Übersicht
df_quadruples = pd.DataFrame(results)

# Sortiere nach n
df_quadruples = df_quadruples.sort_values('n').reset_index(drop=True)

print("\nErste 50 Quadrupel:")
print(df_quadruples.head(50).to_string())

### Analyse der Quadrupel

In [ ]:
# Statistiken
print(f"\nGesamtanzahl Quadrupel: {len(results):,}")

if 'HAS_PANDAS' in globals() and HAS_PANDAS:
    df_quadruples = pd.DataFrame(results)
    print(f"\nKleinste und größte Produkte P:")
    print(f"  Kleinste: P = {df_quadruples['P'].min()}")
    print(f"  Größte: P = {df_quadruples['P'].max()}")
    
    print(f"\nErste 20 Quadrupel (sortiert nach P):")
    for idx, row in df_quadruples.head(20).iterrows():
        print(f"  P={row['P']:6d} = {row['p_e']}*{row['p_a']}*{row['p_b']}*{row['p_c']} = {row['e']}^2 + {row['a']}^2 + {row['b']}^2 + {row['c']}^2")
    
    # Zeige einige interessante Beispiele
    print(f"\nBeispiele mit verschiedenen P-Werten:")
    examples = df_quadruples.iloc[::max(1, len(df_quadruples)//10)]
    for idx, row in examples.head(10).iterrows():
        print(f"  P={row['P']:6d} = {row['p_e']}*{row['p_a']}*{row['p_b']}*{row['p_c']} = {row['e']}^2 + {row['a']}^2 + {row['b']}^2 + {row['c']}^2")
else:
    p_values = [r['P'] for r in results]
    print(f"\nKleinste und größte Produkte P:")
    print(f"  Kleinste: P = {min(p_values)}")
    print(f"  Größte: P = {max(p_values)}")
    
    print(f"\nErste 20 Quadrupel (sortiert nach P):")
    for r in results[:20]:
        print(f"  P={r['P']:6d} = {r['p_e']}*{r['p_a']}*{r['p_b']}*{r['p_c']} = {r['e']}^2 + {r['a']}^2 + {r['b']}^2 + {r['c']}^2")
    
    # Zeige einige interessante Beispiele
    print(f"\nBeispiele mit verschiedenen P-Werten:")
    step = max(1, len(results) // 10)
    for i in range(0, min(10, len(results)), step):
        r = results[i]
        print(f"  P={r['P']:6d} = {r['p_e']}*{r['p_a']}*{r['p_b']}*{r['p_c']} = {r['e']}^2 + {r['a']}^2 + {r['b']}^2 + {r['c']}^2")

### Visualisierung der Quadrupel

In [ ]:
# Visualisierung (nur wenn matplotlib verfügbar)
if 'HAS_MATPLOTLIB' in globals() and HAS_MATPLOTLIB:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Extrahiere Daten
    p_values = [r['P'] for r in results]
    p_e_values = [r['p_e'] for r in results]
    p_a_values = [r['p_a'] for r in results]
    p_b_values = [r['p_b'] for r in results]
    p_c_values = [r['p_c'] for r in results]
    e_values = [r['e'] for r in results]
    a_values = [r['a'] for r in results]
    b_values = [r['b'] for r in results]
    c_values = [r['c'] for r in results]
    
    # 1. Verteilung von P (Produkt)
    axes[0, 0].hist(p_values, bins=50, edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('P = p_e * p_a * p_b * p_c')
    axes[0, 0].set_ylabel('Häufigkeit')
    axes[0, 0].set_title('Verteilung von P (Produkt der Primzahlen)')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Verteilung der Primzahlen
    axes[0, 1].hist(p_e_values, bins=30, alpha=0.5, label='p_e (E)')
    axes[0, 1].hist(p_a_values, bins=30, alpha=0.5, label='p_a (A)')
    axes[0, 1].hist(p_b_values, bins=30, alpha=0.5, label='p_b (B)')
    axes[0, 1].hist(p_c_values, bins=30, alpha=0.5, label='p_c (C)')
    axes[0, 1].set_xlabel('Primzahlwert')
    axes[0, 1].set_ylabel('Häufigkeit')
    axes[0, 1].set_title('Verteilung der Primzahlen')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Verteilung der Lagrange-Komponenten
    axes[1, 0].hist(e_values, bins=30, alpha=0.5, label='e')
    axes[1, 0].hist(a_values, bins=30, alpha=0.5, label='a')
    axes[1, 0].hist(b_values, bins=30, alpha=0.5, label='b')
    axes[1, 0].hist(c_values, bins=30, alpha=0.5, label='c')
    axes[1, 0].set_xlabel('Lagrange-Komponente')
    axes[1, 0].set_ylabel('Häufigkeit')
    axes[1, 0].set_title('Verteilung der Lagrange-Komponenten')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. P vs Summe der Quadrate (sollte identisch sein)
    sum_squares = [r['e']**2 + r['a']**2 + r['b']**2 + r['c']**2 for r in results]
    axes[1, 1].scatter(p_values, sum_squares, alpha=0.5, s=10)
    axes[1, 1].plot([min(p_values), max(p_values)], [min(p_values), max(p_values)], 'r--', label='y=x')
    axes[1, 1].set_xlabel('P (Produkt)')
    axes[1, 1].set_ylabel('e² + a² + b² + c²')
    axes[1, 1].set_title('Verifikation: P = e² + a² + b² + c²')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("Visualisierung übersprungen (matplotlib nicht verfügbar)")

### Export der Ergebnisse

In [ ]:
# Speichere Ergebnisse in CSV
output_filename = 'quadruple_familien_eabc.csv'

if 'HAS_PANDAS' in globals() and HAS_PANDAS:
    df_quadruples = pd.DataFrame(results)
    df_quadruples.to_csv(output_filename, index=False)
    print(f"✓ Ergebnisse gespeichert in '{output_filename}'")
    print(f"  {len(df_quadruples):,} Quadrupel gespeichert")
else:
    # Speichere ohne pandas
    import csv
    with open(output_filename, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['P', 'p_e', 'p_a', 'p_b', 'p_c', 'e', 'a', 'b', 'c', 'L', 'check'])
        writer.writeheader()
        writer.writerows(results)
    print(f"✓ Ergebnisse gespeichert in '{output_filename}'")
    print(f"  {len(results):,} Quadrupel gespeichert")

# Zeige Zusammenfassung
print(f"\nZusammenfassung:")
print(f"  Spalten: P, p_e, p_a, p_b, p_c, e, a, b, c, L, check")
print(f"  Format: P = p_e * p_a * p_b * p_c = e^2 + a^2 + b^2 + c^2")
print(f"  wobei p_e in E, p_a in A, p_b in B, p_c in C (Primzahlen)")

## Hoffbauer-Hilfsfunktionen

In [ ]:
# ---------- Hoffbauer-Hilfen ----------
def antiprime_from_gap(p_i, p_next):
    g = int(p_next - p_i)
    return int(p_i + g//2), g

def index_window(i, N, w):
    lo = max(0, i - w)
    hi = min(N-1, i + w)
    return range(lo, hi+1)

## Hauptfunktionen: A_theta und Dirac-Operator

In [ ]:
# ---------- A_theta aus Hoffbauer-Schnitt ----------
def A_theta_hoffbauer(theta, N=200, w=3, mu=0.35, kappa=0.55, eta=0.20,
                      use_family_phase=True, normalize=True):
    P = [int(nth_prime(i+1)) for i in range(N+1)]
    primes = P[:N]
    primes_next = P[1:N+1]

    gaps, antip = [], []
    for i in range(N):
        a_i, g_i = antiprime_from_gap(primes[i], primes_next[i])
        antip.append(a_i)
        gaps.append(g_i)

    lg = np.log(np.array(gaps, dtype=float))
    lg = (lg - lg.mean()) / (lg.std() + 1e-12)

    A = zero_matrix(CC, N, N)

    # Diagonal: Gap-Potential
    for i in range(N):
        A[i,i] = CC(mu * lg[i])

    # Kette: gerichteter Fluss (mit Holonomie theta)
    ph_edge = complex(np.cos(theta), np.sin(theta))
    for i in range(N-1):
        w_i = kappa / np.sqrt(gaps[i] + 1e-12)
        if use_family_phase:
            extra = np.exp(1j * (fam_phase(primes[i]) - fam_phase(primes[i+1])))
        else:
            extra = 1.0
        A[i, i+1] += CC(w_i * ph_edge * extra)
        A[i+1, i] += CC(w_i * np.conjugate(ph_edge) * np.conjugate(extra))

    # Hoffbauer-Fensterkopplungen über Antiprime-Nähe
    antip_arr = np.array(antip, dtype=float)
    scale = np.median(np.abs(np.diff(antip_arr))) + 1e-12

    for i in range(N):
        for j in index_window(i, N, w):
            if j == i:
                continue
            dist = abs(antip_arr[i] - antip_arr[j]) / scale
            wij = eta * np.exp(-dist) / np.sqrt((gaps[i]+1e-12)*(gaps[j]+1e-12))
            if use_family_phase:
                extra = np.exp(1j * (fam_phase(primes[i]) - fam_phase(primes[j])))
            else:
                extra = 1.0
            A[i,j] += CC(wij * ph_edge * extra)

    if normalize:
        nr = float(max(np.abs(np.array(A.list(), dtype=complex))))
        if nr > 0:
            A = (1.0/nr) * A
    return A

def make_chiral_dirac(A):
    N = A.nrows()
    Z = zero_matrix(CC, N, N)
    return block_matrix([[Z, A],
                         [A.conjugate_transpose(), Z]])

def det_phase(D, t=0.0, eps=1e-6):
    n = D.nrows()
    I = identity_matrix(CC, n)
    Z = D - t*I + (CC(0, eps))*I
    val = Z.det()
    return np.angle(complex(val))

def winding_number(N=200, thetas=361, t=0.0, eps=1e-6, w=3, params=None):
    th = np.linspace(0.0, 2.0*np.pi, thetas)
    phases = []
    for theta in th:
        A = A_theta_hoffbauer(theta, N=N, w=w, **(params or {}))
        D = make_chiral_dirac(A)
        phases.append(det_phase(D, t=t, eps=eps))
    phases = np.unwrap(np.array(phases))
    W = (phases[-1] - phases[0])/(2.0*np.pi)
    return float(W)

def spectral_flow(N=200, thetas=361, t=0.0, w=3, params=None):
    th = np.linspace(0.0, 2.0*np.pi, thetas)
    evs = []
    for theta in th:
        A = A_theta_hoffbauer(theta, N=N, w=w, **(params or {}))
        D = make_chiral_dirac(A) - t*identity_matrix(CC, 2*N)
        lam = np.array([float(rr) for rr in D.eigenvalues()])
        lam.sort()
        evs.append(lam)
    evs = np.array(evs)

    sf = 0
    for j in range(evs.shape[1]):
        s = np.sign(evs[:, j])
        s[s == 0] = 1
        flips = np.where(s[1:] != s[:-1])[0]
        for k in flips:
            if evs[k, j] < 0 and evs[k+1, j] > 0: sf += 1
            if evs[k, j] > 0 and evs[k+1, j] < 0: sf -= 1
    return int(sf)

## 1) Robustheits-Grid Berechnung

In [ ]:
# ---------- 1) Robustheits-Grid ----------
N0 = 200
thetas = 361

grid_w  = [2,3,4]
grid_eps = [1e-4, 1e-6, 1e-8]

rows = []
for w in grid_w:
    for eps in grid_eps:
        W = winding_number(N=N0, thetas=thetas, t=0.0, eps=eps, w=w)
        SF = spectral_flow(N=N0, thetas=thetas, t=0.0, w=w)
        rows.append({"N":N0, "w":w, "eps":eps, "W(t=0)":W, "SF(t=0)":SF})

df = pd.DataFrame(rows)
print(df)

## 2) "137-Umfeld"-Diagnose

In [ ]:
# ---------- 2) "137-Umfeld"-Diagnose ----------
# Idee: wir betrachten zwei Cutoffs:
#   A) N=200 (enthält p=137 sicher)
#   B) N=200, aber wir "maskieren" die ersten m Knoten (setzen Kanten dort auf 0),
#      um zu sehen, ob die frühen Strukturen (inkl. 137-Region) topologisch dominieren.
def A_theta_masked(theta, N=200, mask_first=0, **kwargs):
    A = A_theta_hoffbauer(theta, N=N, **kwargs)
    if mask_first <= 0:
        return A
    # maskiere Zeilen/Spalten im A-Block -> entfernt frühe Knoten dynamisch
    for i in range(mask_first):
        for j in range(N):
            A[i,j] = 0
            A[j,i] = 0
    return A

def winding_number_masked(mask_first, N=200, w=3, eps=1e-6, thetas=361):
    th = np.linspace(0.0, 2.0*np.pi, thetas)
    phases = []
    for theta in th:
        A = A_theta_masked(theta, N=N, mask_first=mask_first, w=w)
        D = make_chiral_dirac(A)
        phases.append(det_phase(D, t=0.0, eps=eps))
    phases = np.unwrap(np.array(phases))
    return float((phases[-1]-phases[0])/(2*np.pi))

masks = [0, 20, 40, 60, 80]  # grob: "entferne" frühe Bereiche (137 liegt bei Index ~33)
Wmask = [winding_number_masked(m, N=N0, w=3, eps=1e-6, thetas=thetas) for m in masks]

## Visualisierungen

In [ ]:
# Plot: Sensitivität von W gegen frühe Regionen
plt.figure()
plt.plot(masks, Wmask, marker="o")
plt.xlabel("mask_first (Anzahl entfernter früher Knoten)")
plt.ylabel("Winding W bei t=0")
plt.title("Hoffbauer-Dirac: Sensitivität von W gegen frühe Regionen (inkl. 137)")
plt.grid(True)
plt.show()

In [ ]:
# Plot Robustheits-Grid als Heatmap-ähnliche Darstellung
pivot = df.pivot(index="w", columns="eps", values="W(t=0)")
plt.figure()
plt.imshow(pivot.values, aspect="auto")
plt.xticks(range(len(pivot.columns)), [str(e) for e in pivot.columns])
plt.yticks(range(len(pivot.index)), [str(w) for w in pivot.index])
plt.xlabel("eps")
plt.ylabel("w")
plt.title("W(t=0) über (w, eps)")
plt.colorbar()
plt.show()